# Hello World with LLaMA - Introduction to Large Language Models

## Lab Overview

This lab introduces the basic inference workflow for decoder-only large language models (LLMs).
You will learn how to load a pretrained causal language model, tokenize text, and generate responses.
For this demonstration, we use TinyLlama as the example model.

> **Quick terms:** A **token** is a small text unit (often a subword) represented as an integer **token ID**.  
> **Logits** are raw scores over the vocabulary before softmax. During generation, these scores are converted into probabilities, and the next token is selected or sampled according to the decoding strategy.  
> **Attention** is the mechanism that lets the model relate tokens to one another.  
> In this notebook, the variable `attention_mask` is a helper tensor that tells the model which positions are real tokens and which are padding.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

By the end of this lab, you will:

- Understand how to load and use pre-trained LLM models.
- Learn about tokenization and text preprocessing.
- Experience text generation with transformer models.
- Understand the basic workflow of LLM inference.

---


## 1. Environment Setup

We'll configure our environment for LLM inference using AMD GPU backend for optimal performance.


In [ ]:
# Core libraries for LLM inference
import warnings

import torch
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Tokenizer, LlamaForCausalLM

warnings.filterwarnings("ignore")

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Model Loading and Configuration

Now we'll load a pre-trained LLaMA model. For this lab, we'll use a publicly available model that demonstrates the core concepts.

**Model Loading Process:**

1. **Model Architecture**: Load the LLaMA model with causal language modeling head
2. **Tokenizer**: Load the corresponding tokenizer for text preprocessing
3. **GPU Transfer**: Move model to AMD GPU for efficient inference

**Note**: In a production environment, you would use the official LLaMA models. For this lab, we'll use a compatible model that demonstrates the same concepts.

**`device_map="auto"` and `accelerate`**

Using **`device_map="auto"`** in `from_pretrained()` asks Hugging Face **Transformers** to place layers on devices (GPU/CPU) automatically. That path relies on the **`accelerate`** library (`pip install accelerate`). If you get errors about missing `accelerate` or device placement, use a simple single-device load instead: omit `device_map`, load the model, then call **`model.to(device)`** as in the next cell.

Do **not** combine `device_map="auto"` with an extra `model.to(device)` on the full model unless you know you need it—pick one loading style. For a single AMD GPU, loading then `.to(device)` is usually enough.


In [ ]:
# Load LLaMA model (using a smaller model for demonstration)
# Note: Replace with actual model path or use a publicly available model
try:
    # For this demo, we'll use a smaller compatible model
    # In practice, you would use the official LLaMA model path
    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Using a smaller Llama model for demonstration

    print("Loading model...")
    model = LlamaForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,  # Use half precision for AMD GPU efficiency
    )

    # Move model to AMD GPU or other selected device
    model = model.to(device)
    model.eval()  # Set to evaluation mode

    print(f"Model loaded successfully on {device}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

except Exception as e:
    print(f"Model loading failed: {e}")
    print("Using a fallback smaller model for demonstration...")

    # Fallback to a smaller model that's publicly available
    model_name = "gpt2"
    from transformers import GPT2LMHeadModel

    model = GPT2LMHeadModel.from_pretrained(model_name)
    model = model.to(device)
    model.eval()
    print("Fallback model loaded successfully")

In [ ]:
# Load tokenizer corresponding to the model
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Set padding token if not available
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Tokenizer loaded successfully")
    print(f"Vocabulary size: {len(tokenizer)}")
    print(f"Special tokens: EOS={tokenizer.eos_token}, PAD={tokenizer.pad_token}")

except Exception as e:
    print(f"Tokenizer loading failed: {e}")
    print("Using fallback tokenizer...")

    # Fallback tokenizer
    from transformers import GPT2Tokenizer

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token
    print("Fallback tokenizer loaded successfully")

## 3. Text Tokenization and Processing

Now we'll process input text through tokenization. This converts human-readable text into numerical tokens that the model can understand.

**Tokenization Process:**

1. **Text Input**: Raw text string
2. **Encoding**: Convert text to token IDs using the tokenizer
3. **Tensor Conversion**: Convert to PyTorch tensors
4. **Device Transfer**: Move tensors to AMD GPU

**Key Components:**

- **input_ids**: Numerical representation of text tokens
- **attention_mask**: Indicates which tokens should be attended to
- **return_tensors**: Format of returned data (PyTorch tensors)


In [ ]:
# Process input text through tokenization
prompt = "Hello! Please introduce yourself and explain what you can do."

# Use English prompt for better compatibility with most models
selected_prompt = prompt

print("Original text:")
print(f"'{selected_prompt}'")
print()

# Tokenize the input
input_data = tokenizer(selected_prompt, return_tensors="pt", padding=True, truncation=True, max_length=512)

# Move to AMD GPU or other selected device
input_ids = input_data["input_ids"].to(device)
attention_mask = input_data["attention_mask"].to(device)

print("Tokenized input:")
print(f"Input IDs shape: {input_ids.shape}")
print(f"Input IDs: {input_ids}")
print(f"Attention mask: {attention_mask}")
print()

# Show token-to-text mapping
print("Token breakdown:")
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
for i, (token_id, token) in enumerate(zip(input_ids[0], tokens)):
    print(f"  {i:2d}: {token_id:5d} -> '{token}'")

## 4. Text Generation

Now we'll use the model to generate text based on our input prompt. The model will predict the next tokens autoregressively.

**Generation Process:**

1. **Input Processing**: Feed tokenized input to the model
2. **Forward Pass**: Model computes probability distributions over vocabulary
3. **Token Sampling**: Select next tokens based on probabilities
4. **Autoregressive Generation**: Repeat process for multiple tokens

**Generation Parameters:**

- **max_length**: Maximum total length (input + generated)
- **max_new_tokens**: Maximum new tokens to generate,when both max_length and max_new_tokens are set, this one takes precedence for controlling generation length
- **temperature**: Controls randomness (lower = more deterministic)
- **do_sample**: Whether to use sampling vs greedy decoding
- **pad_token_id**: Token used for padding sequences


In [ ]:
# Generate text using the model
print("Generating text on AMD GPU...")

with torch.no_grad():  # Disable gradient computation for inference
    try:
        # Generate text with controlled parameters
        # Note: sampling makes outputs non-deterministic across runs,
        # if you want deterministic output, set do_sample=False and temperature=0 or set seed for reproducibility
        generated_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=100,  # Maximum total length (input + generated)
            max_new_tokens=50,  # Maximum new tokens to generate
            temperature=0.7,  # Control randomness
            do_sample=True,  # Use sampling instead of greedy
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,  # Reduce repetition
            no_repeat_ngram_size=3,  # Avoid repeating 3-grams
        )

        print("Text generation completed!")
        print(f"Generated tensor shape: {generated_ids.shape}")
        print(f"Generated IDs: {generated_ids}")

    except Exception as e:
        print(f"Generation failed: {e}")
        # Fallback to simpler generation
        generated_ids = model.generate(input_ids, max_length=60, pad_token_id=tokenizer.pad_token_id)
        print("Fallback generation completed!")

## 5. Output Decoding and Analysis

Finally, we'll convert the generated token IDs back to human-readable text and analyze the results.

**Decoding Process:**

1. **Token to Text**: Convert generated IDs back to text using tokenizer
2. **Special Token Handling**: Remove or handle special tokens appropriately
3. **Post-processing**: Clean up the output text
4. **Analysis**: Examine the generation quality and characteristics

**Key Considerations:**

- **skip_special_tokens**: Whether to remove special tokens from output
- **clean_up_tokenization_spaces**: Handle tokenization artifacts
- **Batch processing**: Handle multiple sequences if applicable


In [ ]:
# Decode the generated tokens back to text
print("Decoding generated text...")

# Decode the full sequence (input + generated)
full_output = tokenizer.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)

# Extract only the generated part (remove input prompt)
input_length = input_ids.shape[1]
# For decoder-only causal LMs, the generated sequence usually starts with the input prompt,
# so we slice off the original prompt tokens to view only the newly generated part.
generated_only_ids = generated_ids[:, input_length:]

generated_text = tokenizer.batch_decode(generated_only_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)

print("Results:")
print("=" * 50)
print(f"Original prompt: '{selected_prompt}'")
print("=" * 50)
print(f"Full output: {full_output[0]}")
print("=" * 50)
print(f"Generated text only: '{generated_text[0]}'")
print("=" * 50)

# Analysis
print("\nGeneration Analysis:")
print(f"Input tokens: {input_length}")
print(f"Generated tokens: {generated_only_ids.shape[1]}")
print(f"Total tokens: {generated_ids.shape[1]}")
print(f"Generated text length: {len(generated_text[0])} characters")

# Performance info
print("\nPerformance:")
print(f"Device used: {device}")
print(f"Model dtype: {next(model.parameters()).dtype}")
print("LLM01 completed successfully!")

## Conclusions

### Technical Concepts Learned

- **Transformer Architecture**: Understanding of autoregressive text generation
- **Tokenization Process**: Text to token conversion and vice versa
- **Model Loading**: Loading and configuring pre-trained LLM models
- **Text Generation**: Generating coherent text using sampling strategies
- **Model Inference Pipeline**: Complete workflow from input to output

### Experiment Further

- Change the `temperature` value (0.1 to 2.0) to control randomness
- Adjust `max_new_tokens` for longer/shorter outputs
- Experiment with different prompts
- Try a Chinese prompt if the model you loaded has multilingual capabilities


---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
